# 🕵️‍♂️ Predatory Publishers Detector - Model Training Pipeline

This notebook provides the step-by-step pipeline used to fine-tune the **main DistilBERT model** for classifying academic journals as **Predatory** or **Legitimate** based on their scraped homepage text.

### Pipeline Overview:
1. **Load and Inspect the Dataset**: Reading the raw CSV data.
2. **Preprocess Labels and Text**: Normalizing texts and mapping string targets to binary labels.
3. **Tokenization**: Converting clean text into token encodings compatible with Transformers.
4. **PyTorch Dataset Setup**: Preparing data loaders for PyTorch compatibility.
5. **Fine-Tuning DistilBERT**: Configuring hyper-parameters and running the Hugging Face `Trainer` loop.
6. **Save Fine-Tuned Weights**: Exporting model weights and tokenizer configurations for production deployment.

## 1. Import Required Libraries

In [ ]:
import os
import numpy as np
import pandas as pd
import torch
from torch.utils.data import Dataset
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments
)

## 2. Load the Dataset

Load the training dataset (`Predatory Journals Dataset.csv`).

In [ ]:
dataset_path = "Predatory Journals Dataset.csv"

if not os.path.exists(dataset_path):
    raise FileNotFoundError(f"Dataset not found at {dataset_path}. Please place it in the same directory.")

df = pd.read_csv(dataset_path)
print(f"Loaded dataset with {len(df)} records.")
df.head()

## 3. Preprocess Text and Target Labels

Identify text/label columns automatically and clean string labels (e.g., mapping "Predatory" to `1` and "Legitimate" to `0`).

In [ ]:
def infer_columns(data_frame):
    lower_cols = {c.lower(): c for c in data_frame.columns}
    
    # Infer label column
    label_candidates = [
        name for key, name in lower_cols.items()
        if any(tok in key for tok in ["label", "predatory", "is_predatory", "target", "class"])
    ]
    if not label_candidates:
        raise ValueError("Could not infer label column.")
    label_col = label_candidates[0]
    
    # Infer text column
    text_order = ["text", "clean_text", "description", "abstract", "content", "journal", "title"]
    text_col = None
    for key in text_order:
        for col_lower, orig in lower_cols.items():
            if key in col_lower:
                text_col = orig
                break
        if text_col:
            break
            
    if not text_col:
        raise ValueError("Could not infer text column.")
        
    return text_col, label_col

def prepare_labels(raw_series):
    if np.issubdtype(raw_series.dtype, np.number):
        vals = raw_series.astype(int).values
        if sorted(set(vals)) == [0, 1]:
            return vals
        raise ValueError("Numeric labels must be 0/1.")
        
    lower = raw_series.astype(str).str.lower()
    mapping = {"predatory": 1, "legitimate": 0, "legit": 0, "non-predatory": 0}
    mapped = lower.map(lambda v: mapping.get(v))
    if mapped.isnull().any():
        raise ValueError("Some string labels did not map to Predatory/Legitimate.")
    return mapped.astype(int).values

text_col, label_col = infer_columns(df)
texts = df[text_col].astype(str).tolist()
labels = prepare_labels(df[label_col])

print(f"Inferred Text Column: {text_col}")
print(f"Inferred Label Column: {label_col}")
print(f"Label breakdown: Predatory={sum(labels)}, Legitimate={len(labels) - sum(labels)}")

## 4. Tokenization and PyTorch Dataset Wrapper

Initialize the standard pre-trained `distilbert-base-uncased` tokenizer, process the input texts, and wrap them in a PyTorch-compatible `Dataset` class.

In [ ]:
model_name = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizing dataset... (this may take a minute)")
encodings = tokenizer(texts, truncation=True, padding=True, max_length=256)

class JournalDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx], dtype=torch.long)
        return item

dataset = JournalDataset(encodings, labels)
print("PyTorch Dataset prepared successfully.")

## 5. Load Model and Configure Training Arguments

Load the base pre-trained DistilBERT sequence classifier and define the fine-tuning parameters.

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    id2label={0: "Legitimate", 1: "Predatory"},
    label2id={"Legitimate": 0, "Predatory": 1}
)

output_dir = "./models/bert_journal_classifier"

training_args = TrainingArguments(
    output_dir=output_dir,
    num_train_epochs=2,              # Fine-tune for 2 epochs
    per_device_train_batch_size=8,   # Batch size of 8
    per_device_eval_batch_size=8,
    learning_rate=5e-5,              # Fine-tuning learning rate
    weight_decay=0.01,
    logging_steps=50,
    save_strategy="epoch",
    load_best_model_at_end=False,
    report_to=[]                     # Disable reporting tools like WandB
)

print("Model and Hyper-parameters prepared.")

## 6. Execute Fine-Tuning

Instantiate the Hugging Face `Trainer` and execute the training loop.

In [ ]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

print("Starting fine-tuning...")
trainer.train()

## 7. Save the Fine-Tuned Model Weights

Export the fine-tuned weights and configurations so that the web application can load them natively.

In [ ]:
os.makedirs(output_dir, exist_ok=True)
model.save_pretrained(output_dir)
tokenizer.save_pretrained(output_dir)
print(f"Fine-tuned model successfully saved to: {output_dir}")